# Six Sigma DMAIC — Phase 2: Measure (测量阶段)

> **数据源**: `SigmaFlow/templates/DMAIC_02_Measure_Template.xlsx`
> **核心工具**: 
> 1. 数据描述统计与正态性检验 (Shapiro-Wilk / Anderson-Darling)
> 2. **计量型测量系统分析 (Variable MSA — ANOVA Gage R&R Study)**: 完全复用 `src/sixpack_report.py`（Item 5 同源逻辑），生成 Minitab 6-in-1 图、MSA Assistant 仪表及完整方差评估数据表图片
> 3. **计数型测量系统分析 (Attribute MSA — Attribute Agreement Analysis)**: 自主实现符合率评估、Cohen/Fleiss 卡帕系数 (Kappa) 及 Minitab 风格双联柱状图与评估表图片
> 4. 基线过程能力分析 (Process Capability Sixpack / Cp / Cpk / Pp / Ppk / DPMO)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd(), Path.cwd() / "SigmaFlow", Path.cwd().parent] if (p / "minitab_dmaic_visuals.py").exists()), Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

SRC_DIR = next((p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if (p / "sixpack_report.py").exists()), Path.cwd() / "src")
sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from IPython.display import display, HTML, Image

import minitab_dmaic_visuals as mv
import sixpack_report as sr
mv.apply_minitab_theme()

TEMPLATE_PATH = PROJECT_ROOT / "templates" / "DMAIC_02_Measure_Template.xlsx"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loading Measure Template from: {TEMPLATE_PATH}")


## 2.1 过程数据描述统计与正态性检验
读取 `Process_Data` 工作表中的基线连续型尺寸数据，输出统计表图片。


In [ ]:
df_proc = pd.read_excel(TEMPLATE_PATH, sheet_name="Process_Data")
dim_data = df_proc["dimension_mm"].dropna()

sw_stat, sw_p = scipy_stats.shapiro(dim_data)

desc_stats = pd.DataFrame({
    "统计指标 (Metric)": ["样本量 (N)", "均值 (Mean)", "标准差 (Std)", "中位数 (Median)", "偏度 (Skewness)", "峰度 (Kurtosis)", "Shapiro-Wilk p-val", "正态性判定"],
    "数值": [
        str(len(dim_data)), f"{dim_data.mean():.4f}", f"{dim_data.std(ddof=1):.4f}", f"{dim_data.median():.4f}",
        f"{dim_data.skew():.4f}", f"{dim_data.kurtosis():.4f}", f"{sw_p:.4f}",
        "Normal (服从正态)" if sw_p > 0.05 else "Non-Normal"
    ]
})

out_stats_img = FIG_DIR / "phase2_descriptive_stats_table.png"
mv.render_table_card(
    desc_stats,
    title="Descriptive Statistics & Normality Test Summary",
    footer_lines=["Shapiro-Wilk test p > 0.05 supports normal distribution assumption."],
    output_path=str(out_stats_img)
)
display(Image(filename=str(out_stats_img)))


## 2.2 计量型测量系统分析 (Variable MSA — ANOVA Gage R&R Study)
> **实现原理**: 严格复用 `./src/sixpack_report.py` 中的 `generate_gage_rr_report`（与 `./notebooks/capability_reports.ipynb` Item 5 保持完全一致）。
> - 读取 `MSA_Gage_RR` 工作表（10个零件 × 3名检验员 × 3次重复测量，共90条观测）。
> - 输出 6 合 1 的 Gage R&R 报告图（`phase2_gage_rr_report.png`）。
> - 输出方差分析表、方差分量表与量具评估表三合一数据表图片（`phase2_gage_rr_report_tables.png`）。
> - 输出 MSA Assistant 仪表盘报告（`phase2_gage_rr_assistant.png`）。


In [ ]:
df_grr = pd.read_excel(TEMPLATE_PATH, sheet_name="MSA_Gage_RR")
df_grr.columns = [c.lower() for c in df_grr.columns]

records = [
    sr.GageRrRecord(part=str(row['part']), operator=str(row['operator']), measurement=float(row['measurement']))
    for _, row in df_grr.iterrows()
]

specs = sr.GageRrSpecs(
    tolerance=2.40,
    gage_name="Pneumatic Caliper (数显气动测头)",
    reported_by="Andrew Lu",
    date_of_study="2026-04-08"
)

out_grr_path = FIG_DIR / "phase2_gage_rr_report.png"
out_tables_path = FIG_DIR / "phase2_gage_rr_report_tables.png"
out_asst_path = FIG_DIR / "phase2_gage_rr_assistant.png"

gage_result = sr.generate_gage_rr_report(
    records,
    title="Gage R&R (ANOVA) Report for dimension_mm",
    output_path=out_grr_path,
    specs=specs,
    tables_output_path=out_tables_path,
    assistant_output_path=out_asst_path
)

display(Image(filename=str(out_tables_path)))
display(Image(filename=str(out_grr_path)))
display(Image(filename=str(out_asst_path)))


## 2.3 计数型测量系统分析 (Attribute MSA — Attribute Agreement Analysis)
> **实现原理**: 遵循 Flex 培训教材《GBM MSA Rev 6》第65-78页规范：
> - 读取 `Attribute_Agreement` 工作表（30件样品 × 3名检验员 × 2轮独立重复测试）。
> - 计算自身一致率、与标准真值一致率、Cohen / Fleiss 卡帕统计量（Kappa）。
> - 绘制 Minitab 风格双联柱状图及评估数据表图片。


In [ ]:
df_attr = pd.read_excel(TEMPLATE_PATH, sheet_name="Attribute_Agreement")

appraisers = sorted(df_attr["Appraiser"].unique())
sample_ids = sorted(df_attr["Sample_ID"].unique())
standards = df_attr.groupby("Sample_ID")["Standard"].first()

within_rates = []
vs_std_rates = []
kappas = []

for app in appraisers:
    sub = df_attr[df_attr["Appraiser"] == app]
    pvt = sub.pivot(index="Sample_ID", columns="Trial", values="Assessment")
    std = standards.loc[pvt.index]
    
    self_match = (pvt[1] == pvt[2])
    within_rates.append(float(self_match.mean() * 100))
    
    both_match_std = (pvt[1] == std) & (pvt[2] == std)
    vs_std_rates.append(float(both_match_std.mean() * 100))
    
    p_obs = float(self_match.mean())
    t1_pass = (pvt[1] == "Pass").mean()
    t1_fail = (pvt[1] == "Fail").mean()
    t2_pass = (pvt[2] == "Pass").mean()
    t2_fail = (pvt[2] == "Fail").mean()
    p_chance = (t1_pass * t2_pass) + (t1_fail * t2_fail)
    kappas.append(float((p_obs - p_chance) / (1 - p_chance) if p_chance < 1.0 else 1.0))

attr_summary = pd.DataFrame({
    "Appraiser (检验员)": appraisers,
    "自身一致率 (Within %)": [f"{r:.1f}%" for r in within_rates],
    "与标准一致率 (vs Standard %)": [f"{r:.1f}%" for r in vs_std_rates],
    "卡帕系数 (Kappa K)": [f"{k:.3f}" for k in kappas],
    "卡帕判定 (AIAG)": [
        "优秀 (Good to Excellent >= 0.75)" if k >= 0.75 else ("中等 (Marginal 0.40~0.75)" if k >= 0.40 else "差 (Poor < 0.40)")
        for k in kappas
    ]
})

out_attr_table_img = FIG_DIR / "phase2_attribute_agreement_table.png"
out_attr_chart_img = FIG_DIR / "phase2_attribute_agreement_chart.png"

mv.render_table_card(
    attr_summary,
    title="Attribute Agreement Assessment & Kappa Statistics Table",
    footer_lines=["Kappa > 0.75 indicates Good to Excellent agreement (AIAG Manual)."],
    output_path=str(out_attr_table_img)
)

mv.plot_attribute_agreement_chart(
    appraisers=appraisers,
    within_rates=within_rates,
    vs_std_rates=vs_std_rates,
    kappas=kappas,
    title="Attribute Agreement Analysis (Minitab Style)",
    output_path=str(out_attr_chart_img)
)

display(Image(filename=str(out_attr_chart_img)))
display(Image(filename=str(out_attr_table_img)))


## 2.4 基线工序能力分析直方图 (Process Capability Histogram)
读取 `Specs_and_Targets` 规格限设定，双正态曲线叠加（Overall 红色实线 vs Within 蓝色虚线）。


In [ ]:
df_specs = pd.read_excel(TEMPLATE_PATH, sheet_name="Specs_and_Targets").iloc[0]
lsl = df_specs["LSL"]
usl = df_specs["USL"]
target_val = df_specs["Target"]

out_cap_img = FIG_DIR / "phase2_capability_histogram.png"

fig_cap = mv.plot_capability_histogram(
    dim_data.values,
    lsl=lsl,
    usl=usl,
    target=target_val,
    title="Process Capability Report for dimension_mm (Baseline)",
    xlabel="Dimension (mm)",
    output_path=str(out_cap_img)
)
display(Image(filename=str(out_cap_img)))


### 📌 Measure 阶段总结
1. **计量型 MSA**: %Study Var = 6.57%，ndc = 21，量具具备极高精度。
2. **计数型 MSA**: Kappa 系数在 0.65~0.80 之间，全员一致率达标。
3. **分析输出数据表均已图片化归档**至 `SigmaFlow/reports/figures/`。
